# Study 958 — Spot ETF Basis 🧊

**After the January-2024 spot bitcoin ETFs, is the futures-vs-spot carry still there?**

Until 2024 the only US-listed, 40-Act routes to bitcoin were a *futures* ETF (BITO, from
October 2021) and a closed-end *trust* (GBTC). Both charged their holders a wedge: the
futures wrapper paid a rich CME basis every day it held the front contract, the trust
swung between a +40% premium and a −49% discount. The tidy story says the spot ETFs, which
create and redeem physical bitcoin at NAV, should have **compressed** both.

We test that on **BITO vs BTC-USD** (2021-10-20 → 2026-06-30, 1,177 sessions)
with **IBIT** and **FBTC** as independent spot rulers from 2024-01-11. Daily
**total-return** closes — BITO's distributions are reinvested; price-only would invent
most of the "drag".

*Numbers below are the frozen headline (`docs/results.md`, Fingerprint `b7f384a06a79`); the
live cells run the fast synthetic control. As-of 2026-06-30.*


## 1. What a wrapper costs you

An ETF that cannot legally hold bitcoin buys *next month's* bitcoin instead. Next month's bitcoin has been persistently more expensive than today's — that gap is the **basis** — and every month the fund watches the gap melt away and then pays it again. The bill shows up as a slow, silent bleed against the coin. Here is what that bill actually was.

In [1]:
R = dict(post_trend=-5.972, post_t=-29.34, vs_ibit=-5.722, vs_ibit_t=-30.68,
         post_mo_t=-2.17, vs_ibit_mo_t=-9.83,
         post_basis=9.44, post_excess=5.02, fee_bito=0.95)
print('BITO vs bitcoin, since the spot ETFs launched: %+.2f%%/yr (t = %.1f, or %.1f on one reading a month)'
      % (R['post_trend'], R['post_t'], R['post_mo_t']))
print('BITO vs IBIT, same trading hours          : %+.2f%%/yr (t = %.1f, or %.1f on one reading a month)'
      % (R['vs_ibit'], R['vs_ibit_t'], R['vs_ibit_mo_t']))
print()
print('of which the fund fee is only %.2f%% -- the rest is the futures carry'
      % R['fee_bito'])
print('implied basis %+.1f%%/yr, i.e. %+.1f%%/yr ON TOP of cash'
      % (R['post_basis'], R['post_excess']))

BITO vs bitcoin, since the spot ETFs launched: -5.97%/yr (t = -29.3, or -2.2 on one reading a month)
BITO vs IBIT, same trading hours          : -5.72%/yr (t = -30.7, or -9.8 on one reading a month)

of which the fund fee is only 0.95% -- the rest is the futures carry
implied basis +9.4%/yr, i.e. +5.0%/yr ON TOP of cash


So the carry is emphatically **still there**: a bitcoin dollar held through the futures wrapper loses about **six cents a year** to a bitcoin dollar held through the spot wrapper. The spot ETFs did not make that go away.

The second line is the one to trust. Comparing a fund to the *coin* means comparing a 4pm New York price to a midnight UTC one; comparing it to another fund does not. Measured fund-against-fund, the gap survives even the bluntest possible test — take one reading a month, 29 readings, no clever statistics — at *t* = −9.8. Against the coin the same blunt test reads only −2.2.

## 2. Can we trust the ruler? Ask it to measure something we already know

Comparing a fund to bitcoin is harder than it sounds: bitcoin trades 24/7 and the price we quote is stamped at midnight UTC, while the funds stop trading at 4pm in New York. Sixteen hours of bitcoin is a *lot* of noise.

The fix is to measure the gap as a **trend line** through every day, instead of just comparing the first and last day. And the way to check it works is to point it at a cost we already know exactly — a spot ETF's published fee.

In [2]:
cal = [('IBIT', -0.25, -0.416, 0.25), ('FBTC', -0.205, -0.459, 0.25)]
for tk, trend, naive, fee in cal:
    print('%s: trend-line reads %+.3f%%/yr | first-vs-last reads %+.3f%%/yr | published fee %+.2f%%/yr'
          % (tk, trend, naive, -fee))

IBIT: trend-line reads -0.250%/yr | first-vs-last reads -0.416%/yr | published fee -0.25%/yr
FBTC: trend-line reads -0.205%/yr | first-vs-last reads -0.459%/yr | published fee -0.25%/yr


> 🔬 **For the quants.** Because daily log differences telescope, `252 × mean(daily diff)` *is* the two-endpoint estimator — it discards every observation in between and inherits both endpoints' intraday offset. The trend slope uses all *n* points. On the real tape it reads IBIT's 25 bp fee to the basis point (*t* = −5.1) where the endpoint estimator cannot tell it from zero (*t* = −0.07).

## 3. The actual question: did the launch change anything?

Split the whole sample at 2024-01-11 and the answer looks dramatic — and backwards. The bleed *widened*, from -2.56%/yr to -5.97%/yr.

But that comparison is rigged. The pre-launch window is mostly **2022**, the crypto bear market, when nobody wanted levered bitcoin and the futures curve barely sloped at all. Compare like with like — the twelve months either side of the launch — and the answer is a flat nothing.

One caveat we owe you: twelve months is *a* choice, and it happens to be the flattest one. Six, nine, eighteen and twenty-four months either side all say the bleed got **worse**, not better. So the honest summary is not "nothing changed" but "nothing got cheaper" — which is still the opposite of what the tidy story predicted.

In [3]:
print('full-sample split : %+.2f%%/yr before  ->  %+.2f%%/yr after   (change %+.2f pp)'
      % (-2.564, -5.969, -3.405))
print('matched 12 months : %+.2f%%/yr before  ->  %+.2f%%/yr after   (change %+.2f pp, t = %.2f)'
      % (-7.316, -7.351, -0.035, -0.08))
print()
print('and the calendar years either side of the launch:')

full-sample split : -2.56%/yr before  ->  -5.97%/yr after   (change -3.40 pp)
matched 12 months : -7.32%/yr before  ->  -7.35%/yr after   (change -0.04 pp, t = -0.08)

and the calendar years either side of the launch:


In [4]:
for y, row in {2022: (-2.61, -3.42, None, 1.43, 3.09), 2023: (-7.08, -24.39, None, 5.01, 11.14), 2024: (-7.41, -21.57, -7.18, 5.2, 11.66), 2025: (-5.31, -30.41, -5.08, 4.15, 8.51), 2026: (-3.13, -7.8, -3.07, 3.49, 5.67)}.items():
    tag = ' <- launch year' if y == 2024 else ''
    part = ' (H1 only)' if y == 2026 else ''
    print('%d: drag %+6.2f%%/yr   implied basis %+6.2f%%/yr%s%s'
          % (y, row[0], row[4], tag, part))

2022: drag  -2.61%/yr   implied basis  +3.09%/yr
2023: drag  -7.08%/yr   implied basis +11.14%/yr
2024: drag  -7.41%/yr   implied basis +11.66%/yr <- launch year
2025: drag  -5.31%/yr   implied basis  +8.51%/yr
2026: drag  -3.13%/yr   implied basis  +5.67%/yr (H1 only)


**2023 and 2024 are the same number.** The wrapper wedge that the spot ETFs *did* close was the other one — GBTC's discount, which snapped to NAV within weeks (see [study 618](../../618-gbtc-premium-cycle/)). The futures basis was never an access wedge; it is a **financing** rate, and it is priced by how badly levered longs want exposure. That is why it is fat in the 2023-24 melt-up and thin in the 2022 and 2026 drawdowns.

## 4. The trap we nearly fell into

That full-sample break had a *t* of -9.50 — hugely 'significant'. So we ran the identical test at 44 **arbitrary** dates that mean nothing. The median arbitrary date produces |*t*| = 4.48, and the launch ranks only **7 of 44**. A slowly wandering series manufactures a 'significant' before/after break almost anywhere you cut it. The event date was not special.

## 5. Live check — the machinery is unbiased (offline synthetic)

**This cell uses synthetic data, not the real tape.** We build a toy world with a known wrapper fee, a known futures carry and the same midnight-vs-4pm timestamp problem, then plant a compression at the event date. The detector must find it — and must stay quiet when the carry is large but unchanged.

In [5]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from etf_basis import data, strategy as st
planted = st.synthetic_detect(*data.synthetic_panel(signal_strength=1.0, seed=958))
null    = st.synthetic_detect(*data.synthetic_panel(signal_strength=0.0, seed=958))
print('SYNTHETIC world, compression planted at %+.2f pp -> detector reads %+.2f pp (t = %+.1f)'
      % (planted['planted_change_pct'], planted['era_change_pct'], planted['era_t']))
print('SYNTHETIC world, NO compression (same big carry) -> detector reads %+.2f pp (t = %+.2f)'
      % (null['era_change_pct'], null['era_t']))
print('SYNTHETIC ruler on a planted %.2f%%/yr fee -> reads %+.3f%%/yr'
      % (null['planted_fee_pct'], null['spot_etf_drag_pct']))

SYNTHETIC world, compression planted at +7.00 pp -> detector reads +7.31 pp (t = +33.2)
SYNTHETIC world, NO compression (same big carry) -> detector reads +0.31 pp (t = +1.43)
SYNTHETIC ruler on a planted -0.25%/yr fee -> reads -0.285%/yr


## 6. Is there money in it?

If the futures wrapper bleeds ~6%/yr against the spot wrapper, buy one and short the other. It works — while your broker lends you BITO cheaply. At a 2% borrow fee and 5 bps of trading cost the pair nets **+3.28%/yr** with a Sharpe of 1.72 and a worst loss of 0.67%. At a 5% borrow fee it nets **+0.30%/yr** — nothing. And it is fading: +4.50% in 2024, +3.22% in 2025, +0.99% in the first half of 2026.

The version that always works needs no borrow at all: **own the spot wrapper instead of the futures one.** That is not alpha — it is not paying a bill you do not have to pay.

## Verdict

- **Signal — Mixed.** The carry is unambiguously alive (-5.72%/yr fund-against-fund, which survives even a one-reading-a-month test at *t* = -9.83; +5.0%/yr above cash), on a ruler that reads a known 25 bp fee exactly. But the claim under test — that the spot ETFs compressed it — is **rejected**: matched twelve-month windows differ by -0.04 pp (*t* = -0.08), and the impressive full-sample break has the wrong sign and ranks 7/44 against placebo dates.
- **Tradability — Fragile.** Long spot / short futures nets +3.28%/yr at 2% borrow, dies at 5%, and is shrinking year by year. Bankable only for someone with cheap, reliable borrow — and the risk-free alternative (owning the spot wrapper) is cost avoidance, not an edge.